# Lab 02 — Data Acquisition

**Course:** AIC365 – Natural Language Processing Fundamentals  
**Objective:** Build the practical skill of getting text out of the real-world formats it lives in — PDFs, Word documents, JSON APIs, and HTML pages.

---

### Lab Task 1
**Task:** From a multi-page PDF, extract text from all pages, clean the text (remove commas, full-stops, quotations, hyphens, dashes), combine it into one string and report the 10 most frequent words.

In [7]:
import PyPDF2
import re
from collections import Counter

# Step 1: Extract text from all pages of a multi-page PDF
try:
    with open("AIC365_NLP_Lab Manual_V1.0.pdf", "rb") as file:
        reader = PyPDF2.PdfReader(file)
        full_text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + " "

    print(f"Total pages processed: {len(reader.pages)}")
    print(f"Total characters extracted (raw): {len(full_text)}")

    # Step 2: Clean the text — remove commas, full-stops, quotations, hyphens, dashes
    cleaned_text = re.sub(r'[,.\'\"\-\u2014\u2013]', '', full_text)

    # Step 3: Convert to lowercase and split into words
    words = cleaned_text.lower().split()
    print(f"Total words after cleaning: {len(words)}")

    # Step 4: Count word frequencies and report the 10 most frequent
    word_counts = Counter(words)
    top_10 = word_counts.most_common(10)

    print("\n--- Top 10 Most Frequent Words ---")
    for rank, (word, count) in enumerate(top_10, 1):
        print(f"  {rank:>2}. '{word}' \u2014 {count} occurrences")

except FileNotFoundError:
    print("PDF file not found. Please ensure the file is in the current directory.")

Total pages processed: 202
Total characters extracted (raw): 393843
Total words after cleaning: 52319

--- Top 10 Most Frequent Words ---
   1. 'the' — 2459 occurrences
   2. 'a' — 1438 occurrences
   3. 'and' — 1251 occurrences
   4. '=' — 759 occurrences
   5. 'to' — 674 occurrences
   6. 'is' — 673 occurrences
   7. 'in' — 667 occurrences
   8. 'of' — 642 occurrences
   9. 'for' — 607 occurrences
  10. 'from' — 379 occurrences


### Lab Task 2
**Task:** Open a multi-paragraph .docx file, extract all paragraphs/lines into a list, then build a DataFrame with one row per paragraph. Add `word_count` and `char_count` columns, and sort the DataFrame by `word_count` to find the 3 longest paragraphs.

In [8]:
import docx
import pandas as pd

# Load the Word document created in Activity 2
try:
    doc = docx.Document("sample.docx")

    # Extract all non-empty paragraphs into a list
    paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]

    # Build a DataFrame with one row per paragraph
    df = pd.DataFrame({
        'paragraph': paragraphs
    })

    # Add word_count and char_count columns
    df['word_count'] = df['paragraph'].apply(lambda p: len(p.split()))
    df['char_count'] = df['paragraph'].apply(len)

    print("--- Full DataFrame ---")
    print(df.to_string(max_colwidth=60))

    # Sort by word_count descending to find the 3 longest paragraphs
    top_3 = df.sort_values('word_count', ascending=False).head(3)

    print("\n--- Top 3 Longest Paragraphs (by word count) ---")
    for idx, row in top_3.iterrows():
        preview = row['paragraph'][:80] + '...' if len(row['paragraph']) > 80 else row['paragraph']
        print(f"  Row {idx}: {row['word_count']} words, {row['char_count']} chars")
        print(f"    Text: {preview}")

except FileNotFoundError:
    print("File 'sample.docx' not found. Run Activity 2 first to create it.")

--- Full DataFrame ---
                                                     paragraph  word_count  char_count
0                         Natural Language Processing Overview           4          36
1  Natural Language Processing (NLP) is a subfield of artif...          32         226
2                                             Key Applications           2          16
3  NLP is used in sentiment analysis, machine translation, ...          25         194
4                                            Challenges in NLP           3          17
5  Ambiguity, sarcasm, context-dependence, and language div...          21         186
6                                            Future Directions           2          17
7  Large Language Models (LLMs) like GPT and BERT have revo...          30         216

--- Top 3 Longest Paragraphs (by word count) ---
  Row 1: 32 words, 226 chars
    Text: Natural Language Processing (NLP) is a subfield of artificial intelligence that ...
  Row 7: 30 words, 21

### Lab Task 3
**Task:** Fetch https://dummyjson.com/users, parse the users array into a DataFrame using the fields `firstName`, `age`, `address.state`, `company.department`. Group the DataFrame by `company.department` and report the average age per department, sorted descending.

In [9]:
import requests
import pandas as pd

# Fetch users from the API
url = "https://dummyjson.com/users"
response = requests.get(url)
data = response.json()

# Parse the users array
users = data['users']

# Extract the required fields into a list of dicts
records = []
for user in users:
    records.append({
        'firstName': user['firstName'],
        'age': user['age'],
        'address_state': user['address']['state'],
        'company_department': user['company']['department']
    })

# Build the DataFrame
df = pd.DataFrame(records)
print("--- Users DataFrame (first 10 rows) ---")
print(df.head(10).to_string(index=False))

# Group by company_department and report average age, sorted descending
avg_age_by_dept = (
    df.groupby('company_department')['age']
    .mean()
    .sort_values(ascending=False)
    .round(1)
)

print("\n--- Average Age per Department (sorted descending) ---")
for dept, avg in avg_age_by_dept.items():
    print(f"  {dept:30s} \u2192 {avg:.1f} years")

--- Users DataFrame (first 10 rows) ---
firstName  age address_state       company_department
    Emily   29   Mississippi              Engineering
  Michael   36       Alabama                  Support
   Sophia   43       Alabama Research and Development
    James   46  Pennsylvania                  Support
     Emma   31      Colorado          Human Resources
   Olivia   23     Tennessee       Product Management
Alexander   39      Delaware              Engineering
      Ava   28  Rhode Island                Marketing
    Ethan   34     Louisiana                  Support
 Isabella   32          Utah                Marketing

--- Average Age per Department (sorted descending) ---
  Research and Development       → 43.0 years
  Sales                          → 40.0 years
  Support                        → 39.2 years
  Engineering                    → 34.0 years
  Services                       → 34.0 years
  Legal                          → 32.3 years
  Training                       →

### Lab Task 4
**Task:** Fetch and parse https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations) with BeautifulSoup, extract the population table into a DataFrame. Clean the population column (strip commas, convert to integer), and report the top 10 most populous countries.

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Fetch the Wikipedia page (User-Agent header required for Wikipedia)
url = "https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) NLP-Lab-Student/1.0"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Find the main population table (it's a wikitable sortable)
table = soup.find('table', class_='wikitable')

if table is None:
    print("Could not find the population table on the page.")
else:
    # Extract table headers
    headers_list = []
    header_row = table.find('tr')
    for th in header_row.find_all('th'):
        headers_list.append(th.get_text(strip=True))

    print("Table Headers:", headers_list)

    # Extract table rows
    rows_data = []
    for row in table.find_all('tr')[1:]:
        cols = row.find_all(['td', 'th'])
        row_data = [col.get_text(strip=True) for col in cols]
        if len(row_data) >= 3:
            rows_data.append(row_data)

    # Build DataFrame
    num_cols = len(rows_data[0]) if rows_data else 0
    col_names = headers_list[:num_cols] if len(headers_list) >= num_cols else [f"col_{i}" for i in range(num_cols)]
    df = pd.DataFrame(rows_data, columns=col_names)

    print(f"\nTotal rows extracted: {len(df)}")
    print("\nFirst 5 rows (raw):")
    print(df.head().to_string())

    # Use the second column as population (Population 1 July 2023 or similar)
    pop_col = df.columns[1]
    country_col = df.columns[0]

    print(f"\nCountry column: '{country_col}'")
    print(f"Population column: '{pop_col}'")

    # Clean the population column: strip commas and convert to integer
    df['population_clean'] = (
        df[pop_col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.extract(r'(\d+)', expand=False)
    )
    df['population_clean'] = pd.to_numeric(df['population_clean'], errors='coerce')

    # Sort by population descending and report top 10
    top_10 = df.nlargest(10, 'population_clean')

    print("\n--- Top 10 Most Populous Countries ---")
    for rank, (_, row) in enumerate(top_10.iterrows(), 1):
        country = row[country_col]
        pop = row['population_clean']
        print(f"  {rank:>2}. {country:30s} | Population: {pop:>15,.0f}")

Table Headers: ['Country or territory', 'Population(1 July 2022)', 'Population(1 July 2023)', 'Change(%)', 'UN continentalregion[1]', 'UN statisticalsubregion[1]']

Total rows extracted: 238

First 5 rows (raw):
  Country or territory Population(1 July 2022) Population(1 July 2023) Change(%) UN continentalregion[1] UN statisticalsubregion[1]
0                World           8,021,407,192           8,091,734,930    +0.88%                       –                          –
1                India           1,425,423,212           1,438,069,596    +0.89%                    Asia              Southern Asia
2             China[a]           1,425,179,569           1,422,584,933    −0.18%                    Asia               Eastern Asia
3        United States             341,534,046             343,477,335    +0.57%                Americas           Northern America
4            Indonesia             278,830,529             281,190,067    +0.85%                    Asia         South-eastern A

### Lab Task 5
**Task:** Scrape https://quotes.toscrape.com/tag/love/ for quote text and author, storing the results as a list of dictionaries. Find which author appears most often, and compute the average quote length (in characters) across all scraped quotes. Save the raw scraped list to a .json file.

In [11]:
import requests
from bs4 import BeautifulSoup
import json
from collections import Counter

# Fetch the quotes page
url = "https://quotes.toscrape.com/tag/love/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Extract quote text and author for each quote
quote_elements = soup.find_all('div', class_='quote')
quotes_list = []

for quote_div in quote_elements:
    text = quote_div.find('span', class_='text').get_text(strip=True)
    author = quote_div.find('small', class_='author').get_text(strip=True)
    quotes_list.append({
        'quote': text,
        'author': author
    })

print(f"Total quotes scraped: {len(quotes_list)}")

# Display all scraped quotes
print("\n--- Scraped Quotes ---")
for i, q in enumerate(quotes_list, 1):
    preview = q['quote'][:70] + '...' if len(q['quote']) > 70 else q['quote']
    print(f"  {i}. \"{preview}\" \u2014 {q['author']}")

# Find which author appears most often
author_counts = Counter(q['author'] for q in quotes_list)
most_common_author, most_common_count = author_counts.most_common(1)[0]
print(f"\nMost frequent author: {most_common_author} ({most_common_count} quotes)")

# Compute the average quote length (in characters)
avg_length = sum(len(q['quote']) for q in quotes_list) / len(quotes_list)
print(f"Average quote length: {avg_length:.1f} characters")

# Save the raw scraped list to a .json file
with open('love_quotes.json', 'w', encoding='utf-8') as f:
    json.dump(quotes_list, f, indent=2, ensure_ascii=False)

print("\nQuotes saved to 'love_quotes.json' successfully!")

Total quotes scraped: 10

--- Scraped Quotes ---
  1. "“It is better to be hated for what you are than to be loved for what y..." — André Gide
  2. "“This life is what you make it. No matter what, you're going to mess u..." — Marilyn Monroe
  3. "“You may not be her first, her last, or her only. She loved before she..." — Bob Marley
  4. "“The opposite of love is not hate, it's indifference. The opposite of ..." — Elie Wiesel
  5. "“It is not a lack of love, but a lack of friendship that makes unhappy..." — Friedrich Nietzsche
  6. "“I love you without knowing how, or when, or from where. I love you si..." — Pablo Neruda
  7. "“If you can make a woman laugh, you can make her do anything.”" — Marilyn Monroe
  8. "“The real lover is the man who can thrill you by kissing your forehead..." — Marilyn Monroe
  9. "“Love does not begin and end the way we seem to think it does. Love is..." — James Baldwin
  10. "“There is nothing I would not do for those who are really my friends. ..." — Jane 

---
*End of Lab 02*